# 06 Sync moderation actions (closed loop)

Moderators act in the `game-cx-players-console` app; each action is written to Lakebase table `game_cx_players.lb_moderation_actions` (Postgres, transactional, ~28 ms round-trip).
This notebook copies those actions back into the lakehouse as Delta table `bronze_app_moderation_actions`, so the pipeline can add them to `gold_fact_event` as `ACTION` rows
and the metric view, dashboard, Genie and the next model retrain all see what moderators did.

* **Source of truth:** Lakebase. The table is small, so each run takes a full snapshot (idempotent; Delta history keeps every prior version).
* **Auth:** short-lived Lakebase OAuth credential minted for the identity running the notebook (no passwords, nothing stored).
* **Runs in:** daily job `game-cx-players-daily-job`, after scoring and before the pipeline refresh.

In [0]:
%pip install -q "psycopg[binary]>=3.1"
%restart_python

In [0]:
import psycopg
from datetime import datetime, timezone
from databricks.sdk import WorkspaceClient
from pyspark.sql import types as T

CATALOG, SCHEMA = "mg_aws_serverless_workspace_catalog", "game_cx_players"
TARGET = f"{CATALOG}.{SCHEMA}.bronze_app_moderation_actions"
ENDPOINT = "projects/game-cx-players-lakebase/branches/production/endpoints/primary"
PG_HOST = "ep-round-unit-d2qva1gu.database.us-east-1.cloud.databricks.com"

w = WorkspaceClient()


def read_lakebase_actions() -> list[tuple]:
    """Return every row of game_cx_players.lb_moderation_actions using a short-lived OAuth credential."""
    cred = w.api_client.do("POST", "/api/2.0/postgres/credentials", body={"endpoint": ENDPOINT})
    with psycopg.connect(host=PG_HOST, dbname="databricks_postgres", user=w.current_user.me().user_name,
                         password=cred["token"], sslmode="require") as conn:
        with conn.cursor() as cur:
            cur.execute("""
              SELECT action_id, player_id, action_type, resolved_by, notes, created_at, resolved_at
              FROM game_cx_players.lb_moderation_actions
              ORDER BY action_id""")
            return cur.fetchall()


rows = read_lakebase_actions()
synced_at = datetime.now(timezone.utc)
schema = T.StructType([
    T.StructField("action_id", T.LongType(), False),
    T.StructField("player_id", T.StringType(), False),
    T.StructField("action_type", T.StringType(), False),
    T.StructField("resolved_by", T.StringType(), True),
    T.StructField("notes", T.StringType(), True),
    T.StructField("created_at", T.TimestampType(), True),
    T.StructField("resolved_at", T.TimestampType(), True),
    T.StructField("_synced_at", T.TimestampType(), False),
])
df = spark.createDataFrame([(*r, synced_at) for r in rows], schema)
(df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET))
spark.sql(f"""COMMENT ON TABLE {TARGET} IS
  'Snapshot of Lakebase game_cx_players.lb_moderation_actions (moderator actions taken in game-cx-players-console). Written by notebook 06_sync_moderation_actions; read by gold_fact_event as ACTION rows.'""")
print(f"Lakebase rows: {len(rows)}")

In [0]:
delta = spark.table(TARGET)
delta_count = delta.count()
assert delta_count == len(rows), f"Mismatch: Lakebase {len(rows)} vs Delta {delta_count}"
print(f"OK: {delta_count} actions synced to {TARGET}")
display(delta.groupBy("action_type").count().orderBy("action_type"))